# Behavior Modeling API: BITS Implementation

## Introduction

Tactics2D provides unified reimplementations of a collection of representative traffic participant behavior models to support the development, validation, and testing of Autonomous Driving Systems (ADS) with realistic and scalable traffic interactions. BITS is one of the default behavior models integrated into Tactics2D.

Original paper: [BITS: Bi-level Imitation for Traffic Simulation](https://arxiv.org/abs/2203.17006)
Original code: [rainmaker22/BITS](https://github.com/rainmaker22/BITS) (released as [tbsim](https://github.com/NVlabs/tbsim))

BITS is bi-level: a **spatial planner** first proposes goal candidates from a rasterized bird's-eye view, then an **agent-aware trajectory module** rolls each candidate forward and a scorer picks the best one on likelihood, progress, lane adherence and collision cost. Both networks consume a raster, so a city-scale map costs it no more than a small one.


## Environment Setup

Please install Tactics2D (`pip install 'tactics2d[behavior]'`) or add the Tactics2D source directory to your `PYTHONPATH`. See the [Installation Guide](https://tactics2d.readthedocs.io/en/latest/installation/) for more details. BITS additionally requires `torch` and `torchvision`.

## Model Preparation

The planner and the predictor ship as **two checkpoints**, and `from_trained_planner` takes both. They are not redistributed with Tactics2D.

| File | Role |
|------|------|
| `bits_planner_nusc_resnet50.ckpt` | the released spatial planner (ResNet-50 backbone) |
| `bits_predictor_nusc_resnet18.ckpt` | the released trajectory module (ResNet-18 backbone) |

The checkpoint records its own architecture, trained dynamics constants and compatibility flag, so the two paths are the whole input.


## Dataset Preparation

Tactics2D does not require datasets to be stored in a fixed location. You can place a dataset in any directory and provide its path when parsing it. Adjust the paths below to match your local data layout.

| Dataset | Role here | Rate | Map |
|---------|-----------|------|-----|
| **nuPlan** | the dataset the released weights were trained on, and the one this port's reproduction runs on | 20 Hz | `.gpkg` |
| **WOMD** | the take-over scene the four behavior demos share, so their numbers can be read side by side | 10 Hz | per-scenario, from the tfrecord |
| **highD / inD** | off-domain: a German motorway and a German urban junction, collected by a different group | 25 Hz | Lanelet2 `.osm` |

!!! warning "The model runs on a fixed 100 ms lattice"
    Every behavior model lays a scenario out on a fixed step - BITS's is `step_ms = 100`, and it reads its history window off that lattice. A log recorded at another rate is **resampled onto that lattice automatically** by the runner (`tactics2d.behavior.rolling_utils.to_lattice`), interpolating positions and headings; fed as-is, most of its history frames would simply be missing.

    No 10 Hz off-domain dataset is set up here, so the off-domain examples below are also off-rate ones. That is a gap in the data on hand, not a step the demos skipped.


## Use BITS for Behavior Generation

Both usages below go through the same public API; the notebook only provides glue code.

| Module | Key API |
|--------|---------|
| **Dataset parsers** | `parser.parse_trajectory(...)` -> `(participants, time_range)`; `parser.parse_map(...)` -> `Map` |
| **Behavior model** | `BitsBehaviorModel.from_trained_planner(...)` -> `.predict(...)`, `.rollout(...)` |
| **Rendering** | `BEVCamera` + `MatplotlibRenderer`, driven through `tutorial_common.render_replay_animation` |


In [1]:
from __future__ import annotations

import inspect
import warnings

warnings.filterwarnings("ignore")

import logging

logging.basicConfig(level=logging.WARNING)


import tutorial_common
from tactics2d.behavior.bits import BitsBehaviorModel
from tactics2d.behavior.rolling_utils import to_lattice
from tactics2d.dataset_parser import LevelXParser, NuPlanParser, WOMDParser
from tactics2d.map.parser import OSMParser
from tactics2d.map.element import Map
from tactics2d.map.map_config import HIGHD_MAP_CONFIG, IND_MAP_CONFIG

pygame 2.6.1 (SDL 2.28.4, Python 3.10.20)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
tutorial_common.apply_notebook_style()

In [3]:
# ---- BITS model ----
# Planner and predictor checkpoints; load the two as a pair.
CHECKPOINT_DIR = "../../../checkpoints/bits"
DEVICE = "cuda"  # or "cpu"

model = BitsBehaviorModel.from_trained_planner(
    planner_checkpoint=f"{CHECKPOINT_DIR}/bits_planner_nusc_resnet50.ckpt",
    predictor_checkpoint=f"{CHECKPOINT_DIR}/bits_predictor_nusc_resnet18.ckpt",
    device=DEVICE,
)
config = model.config

print("history_steps:", config.history_steps)
print("future_steps:", config.future_steps)
print("dt:", config.dt, "s  (step_ms:", config.step_ms, "ms)")
print("raster_size:", config.raster_size, "pixels")
print("pixel_size:", config.pixel_size, "m/px")

# ---- Display constants ----
# Shared with the other behavior demos; see tutorial_common.
NUM_SECONDS = 10

history_steps: 10
future_steps: 20
dt: 0.1 s  (step_ms: 100 ms)
raster_size: 224 pixels
pixel_size: 0.5 m/px


## Take-Over Usage

`predict()` replaces the future of **one** vehicle and leaves everybody else on the trajectory the log recorded. It is the call all four Tactics2D behavior models share, and the one the cross-model comparison is run on.

```python
plan = model.predict(participants, map_, frame, agent_ids=[ego_id])[ego_id]
```

`frame` is the newest frame the model conditions on, in milliseconds, and the return value is `{agent_id: Trajectory}`. BITS is the shortest-horizon model of the four - `future_steps=20`, i.e. 2 s - which is the window every demo in this series shares. The prediction is scored against the vehicle's recorded future with `tutorial_common.displacement_errors`.

### Step 1: Select the Vehicle

This demo uses the shared comparison vehicle - the same junction and the same vehicle the LimSim, InterSim and SMART demos take over - so the four numbers can be read side by side.

### Step 2: Predict, and Score Against the Recorded Future

Scored over the shared 2 s horizon, which here is the model's whole roll-out.

### Step 3: Render with BEVCamera and MatplotlibRenderer

The purple gradient is the path the vehicle has already driven, the green one its plan. Everything outside the modelled set keeps its recorded motion.


In [4]:
# ---- Take-over on the scene the four demos share ----
WOMD_ROOT = "../../../data/womd/uncompressed"

parser = WOMDParser()
file_name = tutorial_common.COMPARISON_FILE
folder = f"{WOMD_ROOT}/{tutorial_common.COMPARISON_SPLIT}"
participants, time_range = parser.parse_trajectory(
    tutorial_common.COMPARISON_SCENARIO, file=file_name, folder=folder
)
map_ = parser.parse_map(tutorial_common.COMPARISON_SCENARIO, file=file_name, folder=folder)
participants = to_lattice(participants, config.step_ms)

ego_id = tutorial_common.COMPARISON_EGO
trigger = tutorial_common.COMPARISON_FRAME_MS
recorded = tutorial_common.recorded_future(participants[ego_id].trajectory, trigger)

plan = model.predict(participants, map_, trigger, agent_ids=[ego_id])[ego_id]
ade, fde, matched = tutorial_common.displacement_errors(
    plan, recorded, tutorial_common.COMPARISON_HORIZON_STEPS
)
print(
    f"ego: {ego_id}  |  predicted {len(plan.frames)} steps, "
    f"{min(plan.frames)}-{max(plan.frames)} ms"
)
print(f"  vs the recorded future:  ADE@2s {ade:.3f} m  FDE@2s {fde:.3f} m  ({matched} steps)")

# Write the prediction onto the vehicle and animate it on the recorded scene.
ego = participants[ego_id]
ego.color = tutorial_common.EGO_COLOR
ego.trajectory._history_states = {
    frame: state for frame, state in ego.trajectory.history_states.items() if frame <= trigger
}
ego.trajectory._frames = sorted(ego.trajectory.history_states)
for frame in plan.frames:
    ego.trajectory.add_state(plan.get_state(frame))

playback_frames = [f for f in ego.trajectory.frames if f <= time_range[1]]
ani_takeover = tutorial_common.render_replay_animation(
    participants,
    map_,
    playback_frames,
    ego_id,
    plans={trigger: [(f, plan.get_state(f).x, plan.get_state(f).y) for f in plan.frames]},
    fps=1000.0 / config.step_ms,
    title_prefix="BITS take-over",
)
ani_takeover

ego: 8  |  predicted 20 steps, 1200-3100 ms
  vs the recorded future:  ADE@2s 1.204 m  FDE@2s 2.831 m  (20 steps)


## Closed-Loop Usage

`rollout()` is the other mode: the vehicle is re-planned once per cycle and the first `replan_interval` states of each plan are committed back into its own trajectory, so the next cycle plans from what the previous one committed. Every other participant keeps its recorded motion.

### Step 1: Select the Vehicle

`tutorial_common.select_ego` picks a vehicle present before the warm-up frame that survives into the second half of the scenario; passing an explicit `ego_id` is just as valid.

### Step 2: Replay the Scenario Closed-Loop

```python
result = model.rollout(participants, map_, ego_id, horizon_ms=10000, replan_interval=20)
```

`horizon_ms` is how far the replay runs and `replan_interval` how many states of a plan are committed before re-planning. The result is a `BitsRollingResult`: the replayed vehicle's whole `trajectory`, the `frames` to animate, and the `plans` issued at each cycle.

### Step 3: Render with BEVCamera and MatplotlibRenderer

The same renderer as above. The driver below parses a scenario, replays it and returns the animation.


In [5]:
def run_takeover_scenario(
    parser,
    file_name=None,
    folder=None,
    map_file=None,
    map_folder=None,
    map_path=None,
    map_config=None,
    ego_id=None,
    ego_color="light-pink",
    fps=10,
    num_seconds=NUM_SECONDS,
    resolution=(1200, 800),
    replan_interval=20,
    **parse_kwargs,
):
    print("Parsing scenario ...")
    participants, time_range = parser.parse_trajectory(
        file=file_name, folder=folder, **parse_kwargs
    )
    participants = to_lattice(participants, config.step_ms)

    # Map loading with fallbacks.
    map_ = None
    if hasattr(parser, "parse_map"):
        # Pass what the map parser takes; the scenario's own window is the
        # trajectory's business.
        takes = inspect.signature(parser.parse_map).parameters
        map_ = parser.parse_map(
            **{name: value for name, value in parse_kwargs.items() if name in takes},
            file=map_file if map_file is not None else file_name,
            folder=map_folder if map_folder is not None else folder,
        )
    if map_ is None and map_path is not None:
        print(f"  loading map from {map_path}")
        map_ = OSMParser(lanelet2=True).parse(file_path=map_path, configs=map_config)
    if map_ is None:
        map_ = Map("empty_map", scenario_type="demo")

    print(f"  participants: {len(participants)},  frames: {time_range}")
    if ego_id is None:
        ego_id = tutorial_common.select_ego(participants)
    participants[ego_id].color = ego_color
    print(f"  takeover target: {ego_id}  (color: {ego_color})")

    # The receding-horizon loop lives in the model package
    # (tactics2d.behavior.bits.rolling); the driver only parses and renders.
    result = model.rollout(
        participants,
        map_,
        ego_id,
        horizon_ms=int(num_seconds * 1000),
        replan_interval=replan_interval,
    )
    frames = result.frames
    print(
        f"  MPC: {result.cycles} cycles  |  playback: {len(frames)} frames  "
        f"({frames[0]}-{frames[-1]} ms, ~{(frames[-1] - frames[0]) / 1000:.0f}s)"
    )

    participants[ego_id].trajectory = result.trajectory
    return tutorial_common.render_replay_animation(
        participants,
        map_,
        frames,
        ego_id,
        plans=result.plans,
        resolution=resolution,
        fps=fps,
        title_prefix="BITS takeover",
    )

### Example 1: nuPlan - Pittsburgh (the model's own dataset)

The released weights were trained on nuPlan/nuScenes, and this is a Pittsburgh log with its `.gpkg` map. A multilane urban road at around 15 m/s: the plan stays on the lane it started in, which is what the training distribution looks like.


In [6]:
ani_nuplan = run_takeover_scenario(
    NuPlanParser(),
    file_name="2021.09.13.19.54.06_veh-45_00781_00843.db",
    folder="../../data/nuplan/data/cache/train_pittsburgh",
    map_file="map.gpkg",
    map_folder="../../data/nuplan/maps/us-pa-pittsburgh-hazelwood/9.17.1937",
    ego_id=43,
    fps=10,
)
ani_nuplan

Parsing scenario ...
  participants: 57,  frames: (22133298350, 22133360099)
  takeover target: 43  (color: light-pink)


  MPC: 5 cycles  |  playback: 60 frames  (22133306350-22133312250 ms, ~6s)


### Example 2: nuPlan - Boston Intersection (off-domain, resampled)

A second nuPlan scene, and the one window the other three demos also replay - the longest pass through an intersection in the Boston log, recomputed from the log's `scenario_tag` table at run time rather than stored, because the parser stamps frames relative to `datetime(2021, 1, 1)` **in the local timezone**.

nuPlan is 20 Hz here, so the runner resamples the window onto the model's 100 ms lattice first.


In [7]:
# ---- nuPlan, same four models, a city-scale map ----
NUPLAN_ROOT = "../../data/nuplan"
nuplan_folder = f"{NUPLAN_ROOT}/data/cache/{tutorial_common.NUPLAN_SCENARIO_FOLDER}"

nuplan_parser = NuPlanParser()
nuplan_window = tutorial_common.nuplan_intersection_window(
    f"{nuplan_folder}/{tutorial_common.NUPLAN_SCENARIO_FILE}"
)
nuplan_participants, _ = nuplan_parser.parse_trajectory(
    file=tutorial_common.NUPLAN_SCENARIO_FILE, folder=nuplan_folder, time_range=nuplan_window
)
nuplan_participants = to_lattice(nuplan_participants, config.step_ms)
nuplan_map = nuplan_parser.parse_map(
    file="map.gpkg", folder=f"{NUPLAN_ROOT}/maps/{tutorial_common.NUPLAN_SCENARIO_MAP}"
)
# The runner lays the log onto the model's 100 ms lattice itself (see
# tactics2d.behavior.rolling_utils.to_lattice).

nuplan_ego = tutorial_common.NUPLAN_SCENARIO_EGO
nuplan_participants[nuplan_ego].color = "light-pink"
print(
    f"window {nuplan_window[0]}-{nuplan_window[1]} ms  |  {len(nuplan_participants)} participants  "
    f"|  {len(nuplan_map.lanes)} lanes"
)

nuplan_result = model.rollout(
    nuplan_participants, nuplan_map, nuplan_ego, horizon_ms=NUM_SECONDS * 1000, replan_interval=20
)
print(f"  MPC: {nuplan_result.cycles} cycles  |  playback: {len(nuplan_result.frames)} frames")

nuplan_participants[nuplan_ego].trajectory = nuplan_result.trajectory
ani_nuplan = tutorial_common.render_replay_animation(
    nuplan_participants,
    nuplan_map,
    nuplan_result.frames,
    nuplan_ego,
    plans=nuplan_result.plans,
    resolution=(1200, 800),
    fps=10,
    title_prefix="BITS takeover (nuPlan)",
)
ani_nuplan

window 20572567849-20572584249 ms  |  100 participants  |  3019 lanes


  MPC: 5 cycles  |  playback: 111 frames


### Example 3: highD - Location 1, Recording 01 (off-domain, resampled)

A German motorway at 25 Hz, resampled onto the lattice. This one is genuinely out of distribution, and the animation shows it: the vehicle drifts about a lane away from where it started and ends up crossing the median onto the opposite carriageway. The scorer weights lane adherence heavily, but it only re-ranks the goal candidates the planner proposes - when none of them follows the road, the best of them still does not.


In [8]:
ani_highd = run_takeover_scenario(
    LevelXParser("highD"),
    file_name=11,
    folder="../../data/highD/data",
    map_path="../../data/highD_map/highD_1.osm",
    map_config=HIGHD_MAP_CONFIG["highD_1"],
    ego_id=18,
    fps=1000.0 / config.step_ms,  # the playback frames are on the model's 100 ms lattice
)
ani_highd

Parsing scenario ...
  loading map from ../../data/highD_map/highD_1.osm
  participants: 1776,  frames: (np.int64(40), np.int64(611080))
  takeover target: 18  (color: light-pink)
  MPC: 5 cycles  |  playback: 97 frames  (40-9640 ms, ~10s)


### Example 4: inD - Location 1, Recording 07 (off-domain, resampled)

A German urban junction at 25 Hz, resampled onto the lattice. Same story at a smaller scale: the plan tracks the road for a while and then drifts, because the geometry, the traffic and the speeds are none of them the ones the weights were trained on.


In [9]:
ani_ind = run_takeover_scenario(
    LevelXParser("inD"),
    file_name=7,
    folder="../../data/inD/data",
    map_path="../../data/inD_map/inD_1.osm",
    map_config=IND_MAP_CONFIG["inD_1"],
    ego_id=12,
    fps=1000.0 / config.step_ms,  # the playback frames are on the model's 100 ms lattice
)
ani_ind

Parsing scenario ...
  loading map from ../../data/inD_map/inD_1.osm
  participants: 212,  frames: (np.int64(0), np.int64(1055240))
  takeover target: 12  (color: light-pink)
  MPC: 5 cycles  |  playback: 64 frames  (10300-16600 ms, ~6s)


### Example 5: WOMD - validation_interactive, Scenario 9 (the yielding case)

The second WOMD scene the four demos share, and the mirror image of the take-over above: the ego opens almost stationary on the yielding side of `2608 -> 2478`, so the relation has to become actual braking instead of a no-op obligation. 42 participants and 311 lanes.


In [ ]:
ani_womd_9 = run_takeover_scenario(
    WOMDParser(),
    file_name=tutorial_common.COMPARISON_FILE,
    folder=f"{WOMD_ROOT}/{tutorial_common.COMPARISON_SPLIT}",
    scenario_id=tutorial_common.SECOND_SCENARIO,
    ego_id=tutorial_common.SECOND_EGO,
)
ani_womd_9

Parsing scenario ...
  participants: 42,  frames: (0, 9000)
  takeover target: 2478  (color: light-pink)
  MPC: 5 cycles  |  playback: 91 frames  (0-9000 ms, ~9s)


## Quick Configurations

| Use Case | Key Settings |
|----------|-------------|
| **Fast preview** | `num_samples=1`, `future_steps=20` - one goal candidate instead of eight |
| **Demo defaults** | `future_steps=20`, `history_steps=10`, `raster_size=224`, `pixel_size=0.5`, `num_samples=8` |
| **Wider neighbourhood** | `max_agents` (default 20) and `max_agents_distance` (default 30 m) size what the raster carries |
| **Scorer trade-off** | `lane_weight` and `collision_weight` (both 100) against `likelihood_weight` and `progress_weight` (both 1) decide which candidate wins |

!!! warning "Match the checkpoint"
    Parameters such as the raster size, the pixel size and the horizon are part of the trained weights. Loading a checkpoint does not check them: a `BitsConfig` that disagrees produces output that looks plausible and is not the trained model.
